# 06 — Interpretability and Pharma Deliverables

Grad-CAM and spatial prediction maps.

In [ ]:
from utils import st_helpers as st

ROOT, PHARMA = st.setup_pharma_paths()
st.set_seeds()
print('ROOT:', ROOT)
print('PHARMA:', PHARMA)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import squidpy as sq
from src.data import load_config, cohort_slide_ids, pharma_outputs_dir, load_slide
from src.labels import build_labels_cohort
from src.train import train_loso
from src.eval import grad_cam_for_patch, evaluate_fold

cfg = load_config()
oncology = cfg['cohorts']['oncology']
labels = build_labels_cohort(cohort_slide_ids(cfg), cfg=cfg)
breast_labels = labels[labels['slide_id'].isin(oncology)]
results = train_loso(oncology, breast_labels, cfg=cfg)
ev = evaluate_fold(results[0])


In [ ]:
model, device = results[0]['model'], results[0]['device']
X_val, y_pred = results[0]['X_val'], ev['y_cls_pred']
y_true = ev['y_cls']
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for ax, idx in zip(axes[0], np.where(y_true == y_pred)[0][:3]):
    cam = grad_cam_for_patch(model, X_val[idx], int(y_pred[idx]), device=device)
    ax.imshow((X_val[idx].transpose(1,2,0)*255).astype(np.uint8))
    ax.imshow(cam, cmap='jet', alpha=0.45); ax.set_title('correct'); ax.axis('off')
for ax, idx in zip(axes[1], np.where(y_true != y_pred)[0][:3]):
    cam = grad_cam_for_patch(model, X_val[idx], int(y_pred[idx]), device=device)
    ax.imshow((X_val[idx].transpose(1,2,0)*255).astype(np.uint8))
    ax.imshow(cam, cmap='jet', alpha=0.45); ax.set_title('incorrect'); ax.axis('off')
fig.savefig(pharma_outputs_dir() / 'gradcam_montage.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
val_slide = results[0]['val_slide']
adata = load_slide(val_slide)
adata.obs['pred_cluster'] = 'NA'
adata.obs.loc[results[0]['lab_val']['spot_id'], 'pred_cluster'] = y_pred.astype(str)
sq.pl.spatial_scatter(adata, color='pred_cluster', size=1.3)
plt.savefig(pharma_outputs_dir() / 'spatial_predictions.png', dpi=120, bbox_inches='tight')
plt.show()


Compare the run with the frozen-model findings in [`PROJECT_REPORT.md`](../PROJECT_REPORT.md).

**End of pipeline.**